In [1]:
import numpy as np
import pandas as pd

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("vaibhavkumar11/hindi-english-parallel-corpus")

print("Path to dataset files:", path)



Using Colab cache for faster access to the 'hindi-english-parallel-corpus' dataset.
Path to dataset files: /kaggle/input/hindi-english-parallel-corpus


In [3]:
!cp -r /kaggle/input/hindi-english-parallel-corpus /content

import pandas as pd

df = pd.read_csv("/content/hindi-english-parallel-corpus/hindi_english_parallel.csv")

df.head()

df.describe()

df = df[['english', 'hindi']]
df.dropna(inplace=True)

# Use smaller subset for speed
df = df.sample(80000, random_state=42)

df['english'] = df['english'].str.lower()
df['hindi'] = df['hindi'].str.lower()

df['hindi'] = 'start ' + df['hindi'] + ' end'

In [4]:
df.sample(4)

,english,hindi
1068996,did the energy go?,start ऊर्जा जाना था? end
701996,be aware that abnormal cells can be treated.,start याद रहेः अप्रसामान्य (एबनार्मल) कोशाणुओं...
887664,vigna sinensis,start चवल end
994516,rhinencephalic refers to that belong to rhinen...,start घ्राणमस्तिष्कीय उसे निर्दिष्ट करता है जो...


In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [6]:
MAX_VOCAB = 20000

eng_tokenizer = Tokenizer(num_words=MAX_VOCAB, filters='', oov_token="<unk>")
eng_tokenizer.fit_on_texts(df['english'])

hin_tokenizer = Tokenizer(num_words=MAX_VOCAB, filters='', oov_token="<unk>")
hin_tokenizer.fit_on_texts(df['hindi'])

eng_seq = eng_tokenizer.texts_to_sequences(df['english'])
hin_seq = hin_tokenizer.texts_to_sequences(df['hindi'])

eng_vocab = MAX_VOCAB
hin_vocab = MAX_VOCAB

from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 25

encoder_input = pad_sequences(
    eng_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

decoder_input = pad_sequences(
    hin_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

# Teacher forcing target
decoder_target = np.zeros_like(decoder_input)
decoder_target[:, :-1] = decoder_input[:, 1:]
decoder_target = np.expand_dims(decoder_target, -1)

In [7]:
from sklearn.model_selection import train_test_split
X1_train, X1_val, X2_train, X2_val, y_train, y_val = train_test_split(
    encoder_input,
    decoder_input,
    decoder_target,
    test_size=0.2,
    random_state=42
)

In [8]:
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

In [9]:
embedding_dim = 128

# encoder
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(MAX_VOCAB, embedding_dim, mask_zero=True)(encoder_inputs)
_, state_h, state_c = LSTM(512, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

# decoder — save layers as variables so we can reuse them later
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(MAX_VOCAB, embedding_dim, mask_zero=True)
dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = LSTM(512, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = Dense(MAX_VOCAB, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [10]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │  2,560,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 128) │  2,560,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 512),     │  1,312,768 │ embedding[0][0],  │
│                     │ (None, 512),      │            │ not_equal[0][0]   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │  1,312,768 │ embedding_1[0][0… │
│                     │ 512), (None,      │            │ lstm[0][1],       │
│                     │ 512), (None,      │            │ lstm[0][2]        │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 10,260,000 │ lstm_1[0][0]      │
│                     │ 20000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 18,005,536 (68.69 MB)

 Trainable params: 18,005,536 (68.69 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.fit(
    [X1_train, X2_train],
    y_train,
    validation_data=([X1_val, X2_val], y_val),
    batch_size=32,
    epochs=5
)

Epoch 1/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 187s 91ms/step - accuracy: 0.2153 - loss: 6.0861 - val_accuracy: 0.1932 - val_loss: 5.0595
Epoch 2/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 184s 92ms/step - accuracy: 0.1952 - loss: 4.8080 - val_accuracy: 0.2122 - val_loss: 4.6598
Epoch 3/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 186s 93ms/step - accuracy: 0.2400 - loss: 4.2667 - val_accuracy: 0.3839 - val_loss: 4.4405
Epoch 4/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 190s 95ms/step - accuracy: 0.3079 - loss: 3.7719 - val_accuracy: 0.2900 - val_loss: 4.3473
Epoch 5/5
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 189s 94ms/step - accuracy: 0.3775 - loss: 3.3574 - val_accuracy: 0.3899 - val_loss: 4.3352


In [14]:
# ─── INFERENCE MODELS ───────────────────────────────────

# encoder model — takes english sentence, gives memory
encoder_model = Model(encoder_inputs, encoder_states)

# decoder model — takes one word + memory, gives next word + new memory
h_state_input = Input(shape=(512,))
c_state_input = Input(shape=(512,))

dec_emb2 = dec_emb_layer(decoder_inputs)
dec_out2, h2, c2 = decoder_lstm(dec_emb2, initial_state=[h_state_input, c_state_input])
dec_out2 = decoder_dense(dec_out2)

decoder_model = Model(
    [decoder_inputs, h_state_input, c_state_input],
    [dec_out2, h2, c2]
)

In [20]:
# ─── TRANSLATION ────────────────────────────────────────

hin_index_to_word = {v: k for k, v in hin_tokenizer.word_index.items()}


In [21]:
def translate(english_sentence):
    seq = eng_tokenizer.texts_to_sequences([english_sentence])
    seq = pad_sequences(seq, maxlen=MAX_LEN, padding='post')

    h, c = encoder_model.predict(seq, verbose=0)

    current_token = np.zeros((1, 1))
    current_token[0, 0] = hin_tokenizer.word_index['start']

    result = []

    for _ in range(MAX_LEN):
        predicted, h, c = decoder_model.predict([current_token, h, c], verbose=0)

        word_index = np.argmax(predicted[0, -1, :])
        word = hin_index_to_word.get(word_index, "")

        if word == "end" or word == "":
            break

        result.append(word)
        current_token[0, 0] = word_index

    print("Input  :", english_sentence)
    print("Output :", " ".join(result))


# test
translate("i am going home")
translate("i love my country")
translate("how are you")

Input  : i am going home
Output : मैं मुझे <unk>
Input  : i love my country
Output : मैं तो मुझे <unk>
Input  : how are you
Output : तुम कह रहे हो?
